# Notebook Setup Stuff

In [1]:
# setup the venv
# run this cell first, then refresh the page and select the kernel.
# there is no harm in running this after setting it up, inly a few seconds
!bash /home/hcr64/Pinyon-Detection/shells/jupyter_kernel_env.sh

Installed kernelspec open3d_env in /home/hcr64/.local/share/jupyter/kernels/open3d_env


In [2]:
import sys, numpy as np, pandas as pd
from scipy.spatial import KDTree
from sklearn.cluster import KMeans
import rasterio
from rasterio.enums import Resampling
from affine import Affine

sys.path.insert(0, "/home/hcr64/Pinyon-Detection")
sys.path.insert(0, "/home/hcr64/Pinyon-Detection/clustering")

from global_files import get_paths, load_clusters
from functions.preprocessing.build_chm import load_chm
from functions.detection.find_chm_peaks import find_chm_peaks, chm_peaks_as_kmeans_seeds
from functions.detection.split_large_clusters import filter_cluster

# ── config ────────────────────────────────────────────────────────────────
TRIAL          = "Sunset_sfm_trial"
KNOWN          = [1399]      # <- put your known merged-tree cluster indices here
# FINE_SMOOTH    = 3.0         # production uses 3; lighter smoothing keeps secondary tops
# FINE_SEARCH_M  = 1.5         # production uses 2.5
MIN_HEIGHT     = 0.5
PEAK_TO_PT_MAX = 2.5         # a peak counts as "inside" if a cluster point is this close in XY
MIN_PEAK_SEP_M = 2.0         # drop peaks closer than this (crude stand-in for a prominence filter)
MIN_POINTS     = 240

PATHS    = get_paths(TRIAL)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
d = np.load(PATHS["Dataframes"] + "fine_peaks_s1.5_r1.5.npz")
peak_coords, peak_heights = d["coords"], d["heights"]
print(f"{len(peak_coords)} peaks (sigma={d['smooth_sigma']}, search={d['search_radius_m']} m)")

clusters = load_clusters(PATHS["Clusters"])
print(f"{len(clusters)} clusters")

9730 peaks (sigma=1.5, search=1.5 m)
2015 clusters


# Peak Splitting, No Plateau

### Functions

Peaks/splitting functions

In [4]:
def peaks_in_cluster(pts):
    """Fine-scale peaks that sit on this cluster's points, thinned by min separation."""
    xy = pts[:, :2]
    lo, hi = xy.min(0) - 1, xy.max(0) + 1
    box = np.all((peak_coords >= lo) & (peak_coords <= hi), axis=1)
    cand, h = peak_coords[box], peak_heights[box]
    if len(cand) == 0:
        return cand
    d, _ = KDTree(xy).query(cand)
    cand, h = cand[d <= PEAK_TO_PT_MAX], h[d <= PEAK_TO_PT_MAX]
    # greedy: tallest first, drop anything too close to an accepted peak
    keep = []
    for i in np.argsort(-h):
        if all(np.linalg.norm(cand[i] - cand[j]) >= MIN_PEAK_SEP_M for j in keep):
            keep.append(i)
    return cand[keep]

def test_split(idx, verbose=True):
    pts = np.asarray(clusters[idx].points)
    p = peaks_in_cluster(pts)
    out = {"file": idx, "n_points": len(pts), "n_peaks": len(p), "n_valid_subs": 0}
    if len(p) < 2:
        if verbose: print(f"cluster {idx}: {len(p)} peak(s) -> would exit (one tree)")
        return out
    seeds = chm_peaks_as_kmeans_seeds(p, pts)          # (N, 3), z = cluster mean z
    km = KMeans(n_clusters=len(p), init=seeds[:, :2], n_init=1).fit(pts[:, :2])  # XY only
    subs = [clusters[idx].select_by_index(np.where(km.labels_ == k)[0]) for k in range(len(p))]
    valid = [s for s in subs if len(s.points) >= MIN_POINTS and filter_cluster(s, 1.0, 0.3)]
    out["n_valid_subs"] = len(valid)
    if verbose:
        print(f"cluster {idx}: {len(p)} peaks -> {len(valid)}/{len(subs)} valid sub-clusters")
        for k, s in enumerate(subs):
            sp = np.asarray(s.points); ext = sp.max(0) - sp.min(0)
            print(f"   sub {k}: n={len(sp):>5}  h={ext[2]:.2f}m  r={max(ext[:2])/2:.2f}m  "
                  f"center=({km.cluster_centers_[k][0]:.1f}, {km.cluster_centers_[k][1]:.1f})")
    return out

Drawing Pointclouds

In [5]:
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from plyfile import PlyData

# distinct colours used when color_by_source=True
_PALETTE = ["#e6194b", "#3cb44b", "#4363d8", "#f58231",
            "#911eb4", "#46f0f0", "#f032e6", "#bcf60c"]

def _points_and_colors(src):
    """Return (N,3) points and (N,3) colours in 0-255 (or None) from a path or PointCloud."""
    if isinstance(src, o3d.geometry.PointCloud):
        points = np.asarray(src.points)
        # Open3D stores colours as floats in 0-1
        colors = np.asarray(src.colors) * 255 if src.has_colors() else None
    else:
        vertex = PlyData.read(str(src))["vertex"]
        points = np.vstack([vertex["x"], vertex["y"], vertex["z"]]).T
        names = vertex.data.dtype.names
        if {"red", "green", "blue"} <= set(names):
            # .ply files written by Open3D store colours as 0-255
            colors = np.vstack([vertex["red"], vertex["green"], vertex["blue"]]).T
        else:
            colors = None
    return points, colors

def visualize_pointcloud(src, point_size=2, background_color="black",
                         color_by_source=False, max_points=None):
    """
    src: a .ply path, an o3d PointCloud, or a list/tuple of either.
    color_by_source: give each item in a list one flat colour, to compare split pieces.
    max_points: randomly subsample each cloud to this many points for faster rendering.
    """
    sources = list(src) if isinstance(src, (list, tuple)) else [src]
    rng = np.random.default_rng(0)
    traces = []

    for i, s in enumerate(sources):
        points, colors = _points_and_colors(s)

        if max_points is not None and len(points) > max_points:
            keep = rng.choice(len(points), max_points, replace=False)
            points = points[keep]
            colors = colors[keep] if colors is not None else None

        if color_by_source or colors is None:
            marker_color = _PALETTE[i % len(_PALETTE)] if color_by_source else "white"
        else:
            marker_color = [f"rgb({int(r)},{int(g)},{int(b)})" for r, g, b in colors]

        traces.append(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode="markers",
            marker=dict(size=point_size, color=marker_color),
            name=f"piece {i} ({len(points)} pts)",
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(scene=dict(bgcolor=background_color, aspectmode="data"),
                      paper_bgcolor="black",
                      legend=dict(font=dict(color="white")))
    fig.show()

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.spatial import KDTree

def show_cluster_peaks(idx, max_pts=40000, marker_r=0.5, stem=2.0):
    pcd = clusters[idx]
    pts = np.asarray(pcd.points)
    peaks = peaks_in_cluster(pts)

    # downsample for speed
    sel = np.random.default_rng(0).choice(len(pts), min(len(pts), max_pts), replace=False)
    p = pts[sel]
    if pcd.has_colors():
        col = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
               for r, g, b in np.asarray(pcd.colors)[sel]]
    else:
        col = p[:, 2]

    fig = go.Figure(go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
        marker=dict(size=1.5, color=col, opacity=0.6), name="cluster"))

    # peaks only have XY (CHM heights are above ground, not absolute Z),
    # so take the top Z of the cluster points near each peak
    tree = KDTree(pts[:, :2])
    for k, pk in enumerate(peaks):
        near = tree.query_ball_point(pk, r=marker_r)
        z = pts[near, 2].max() if near else pts[:, 2].max()
        fig.add_trace(go.Scatter3d(
            x=[pk[0], pk[0]], y=[pk[1], pk[1]], z=[z, z + stem],
            mode="lines+markers", line=dict(width=6, color="red"),
            marker=dict(size=[0, 6], color="red"), name=f"peak {k}"))

    fig.update_layout(scene=dict(aspectmode="data"),
                      title=f"cluster {idx}: {len(peaks)} peaks", height=700)
    fig.show()

# show_cluster_peaks(1370)

### Load in Data

In [7]:
# Clusters
clusters = load_clusters(PATHS["Clusters"])

In [8]:
# CHM
#chm, transform, crs = load_chm(PATHS["CHM"])
with rasterio.open(PATHS["CHM"]) as src:
    print(src.width, src.height, src.res)
    print(f"~{src.width * src.height * 4 / 1e9:.1f} GB per float32 copy")

16770 21270 (0.03674, 0.03674)
~1.4 GB per float32 copy


In [9]:
TARGET_RES = 0.1   # metres/px for the notebook test

with rasterio.open(PATHS["CHM"]) as src:
    native = src.res[0]
    f = max(1, round(TARGET_RES / native))
    chm = src.read(1, out_shape=(src.height // f, src.width // f),
                   resampling=Resampling.average).astype("float32")
    transform = src.transform * Affine.scale(f)   # keep pixel->UTM mapping correct
    crs = src.crs

res = abs(transform.a)
print(f"native {native:.3f} m/px -> using {res:.3f} m/px, shape {chm.shape}")

native 0.037 m/px -> using 0.110 m/px, shape (7090, 5590)


In [10]:
# resolution
res = abs(transform.a)
print(f"{len(clusters)} clusters, CHM res {res} m/px")

2015 clusters, CHM res 0.11022000000000001 m/px


### Look For Peaks

In [11]:
# Find Peaks in CHM
peak_coords, peak_heights = find_chm_peaks(
    chm, transform, min_height=MIN_HEIGHT, search_radius_m=FINE_SEARCH_M,
    resolution=res, smooth_sigma=FINE_SMOOTH,
)

NameError: name 'FINE_SEARCH_M' is not defined

In [ ]:
# 1) targeted: your known-problem clusters
for i in KNOWN:
    test_split(i)

# 2) scan everything: how many clusters would this try to split?
scan = pd.DataFrame([test_split(i, verbose=False) for i in range(len(clusters))])
print(f"\nclusters with >=2 fine peaks: {(scan.n_peaks >= 2).sum()} / {len(scan)}")
print(scan.n_peaks.value_counts().sort_index().to_string())

In [ ]:
import numpy as np, pandas as pd

# counts only (no splitting), uses peaks_in_cluster() from the earlier cell
n_peaks = np.array([len(peaks_in_cluster(np.asarray(c.points))) for c in clusters])

multi_idx = np.where(n_peaks > 1)[0].tolist()

print(f"{len(multi_idx)} / {len(clusters)} clusters have >1 peak")
print(pd.Series(n_peaks).value_counts().sort_index().to_string())

# peek at the biggest ones, and where your known cluster lands
top = sorted(multi_idx, key=lambda i: -n_peaks[i])[:15]
print("most peaks:", [(i, int(n_peaks[i])) for i in top])
print("1399 peaks:", int(n_peaks[1399]) if len(n_peaks) > 1370 else "n/a")

# save so you don't have to rescan
pd.DataFrame({"file": multi_idx, "n_peaks": n_peaks[multi_idx]}).to_csv(
    PATHS["Dataframes"] + "multi_peak_clusters.csv", index=False)

In [ ]:
visualize_pointcloud( clusters[ 267
    ])#KNOWN[0]] )

In [ ]:
show_cluster_peaks(267)

# Split Using Peaks and  Plateaus

In [ ]:
def seeds_for_cluster(idx):
    regions = maxima_regions(idx)
    seeds = [np.asarray(s) for r in regions for s in r["seeds"]]
    return np.array(seeds).reshape(-1, 2), regions

def test_split_h(idx, verbose=True):
    pcd = clusters[idx]
    pts = np.asarray(pcd.points)
    seeds, regions = seeds_for_cluster(idx)
    out = {"file": idx, "n_points": len(pts), "n_regions": len(regions),
           "n_plateaus": sum(r["plateau"] for r in regions),
           "n_seeds": len(seeds), "n_valid_subs": 0}
    if len(seeds) < 2:
        if verbose: print(f"cluster {idx}: {len(seeds)} seed(s) -> exit (one tree)")
        return out

    km = KMeans(n_clusters=len(seeds), init=seeds, n_init=1).fit(pts[:, :2])
    subs = [pcd.select_by_index(np.where(km.labels_ == k)[0]) for k in range(len(seeds))]
    valid = [s for s in subs if len(s.points) >= MIN_POINTS and filter_cluster(s, 1.0, 0.3)]
    out["n_valid_subs"] = len(valid)

    if verbose:
        print(f"cluster {idx}: {len(regions)} regions "
              f"({out['n_plateaus']} plateau) -> {len(seeds)} seeds "
              f"-> {len(valid)}/{len(subs)} valid sub-clusters")
        for k, s in enumerate(subs):
            sp = np.asarray(s.points); ext = sp.max(0) - sp.min(0)
            print(f"   sub {k}: n={len(sp):>5}  h={ext[2]:.2f}m  r={max(ext[:2])/2:.2f}m")
    return out, seeds, km, subs if False else out

# targeted
for i in KNOWN:
    test_split_h(i)

In [ ]:
import numpy as np, rasterio
from rasterio.windows import from_bounds
from scipy.ndimage import gaussian_filter, label
from skimage.morphology import h_maxima
from skimage.feature import peak_local_max
from scipy.ndimage import distance_transform_edt
from scipy.spatial import KDTree

H_M            = 0.3    # prominence: a max must rise this far above its saddle (m)
SMOOTH_PX      = 1.0    # light smoothing on the cropped CHM (pixels)
PLATEAU_AREA   = 1.0    # m^2; regions larger than this are flagged as plateaus
PLATEAU_SEP_M  = 2.0    # min spacing between seeds inside one plateau (m)

src = rasterio.open(PATHS["CHM"])

def maxima_regions(idx, pad=1.0):
    pts = np.asarray(clusters[idx].points)
    xy = pts[:, :2]
    x0, y0 = xy.min(0) - pad; x1, y1 = xy.max(0) + pad
    win = from_bounds(x0, y0, x1, y1, src.transform).round_offsets().round_lengths()
    chm = src.read(1, window=win, boundless=True, fill_value=np.nan).astype("float32")
    wt = src.window_transform(win)
    px = abs(wt.a)

    nan = np.isnan(chm)
    sm = gaussian_filter(np.where(nan, 0, chm), SMOOTH_PX)
    sm[nan] = 0

    # h-maxima on integer millimetres (avoids float-equality issues)
    mm = np.round(sm * 1000).astype(np.int32)
    mask = h_maxima(mm, int(H_M * 1000)).astype(bool) & (sm >= 0.5)
    lab, n = label(mask)

    tree = KDTree(xy)
    out = []
    for k in range(1, n + 1):
        rr, cc = np.nonzero(lab == k)
        area = len(rr) * px * px
        xs, ys = rasterio.transform.xy(wt, rr, cc)
        pts_k = np.column_stack([xs, ys])
        cen = pts_k.mean(0)
        if tree.query(cen)[0] > 0.5:          # must sit on this cluster
            continue
        rec = dict(center=cen, area=area, height=float(sm[rr, cc].max()),
                   plateau=area >= PLATEAU_AREA, seeds=[cen])
        if rec["plateau"]:
            # split elongated plateaus into several seeds
            sub = (lab == k)
            dist = distance_transform_edt(sub) * px
            pk = peak_local_max(dist, min_distance=max(1, int(PLATEAU_SEP_M / px)),
                                labels=sub.astype(int), exclude_border=False)
            if len(pk) > 1:
                sx, sy = rasterio.transform.xy(wt, pk[:, 0], pk[:, 1])
                rec["seeds"] = list(np.column_stack([sx, sy]))
        out.append(rec)
    return out

for r in maxima_regions(1399):
    print(f"center=({r['center'][0]:.1f},{r['center'][1]:.1f}) area={r['area']:.2f} m² "
          f"h={r['height']:.1f} plateau={r['plateau']} seeds={len(r['seeds'])}")

In [ ]:
scan_h = pd.DataFrame([test_split_h(i, verbose=False) for i in range(len(clusters))])
multi_h = scan_h.loc[scan_h.n_seeds > 1, "file"].tolist()

print(f"{len(multi_h)} / {len(clusters)} clusters have >1 seed")
print(scan_h.n_seeds.value_counts().sort_index().to_string())
print("plateau clusters:", int((scan_h.n_plateaus > 0).sum()))
scan_h.to_csv(PATHS["Dataframes"] + "h_maxima_scan.csv", index=False)